# CLIP 

In [ ]:
# All libraries used in this notebook have been included hereunder 
import os
from pathlib import Path
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(device)


2.5.1+cu121
cuda


In [2]:
dataset_path = Path(r'C:\Users\danie\Desktop\_\Daniela Curmi\University\Final Year Project\Final Year Project Code Implementation\Website\paintings')

In [4]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## Preprocessing 

### Image Processing and Embedding Extraction

In [ ]:
def load_process_extract_images(dataset_path):
    image_embeddings = []
    image_ids = []

    for root, dirs, files in os.walk(dataset_path):
        for filename in files:
            if not filename.lower().endswith((".jpg", ".png", ".jpeg")):
                continue

            image_path = os.path.join(root, filename)

            try:
                image = Image.open(image_path).convert("RGB")
                inputs = processor(images=image, return_tensors="pt").to(device)
                with torch.no_grad():
                    features = model.get_image_features(**inputs)
                    features = features / features.norm(dim=-1, keepdim=True)

                image_embeddings.append(features.cpu().numpy())
                image_ids.append(filename.id)

            except Exception as e:
                print(f"[WARNING] Skipping {image_path}: {e}")
                continue

    return np.vstack(image_embeddings), image_ids

In [ ]:
image_tensors, image_paths = load_process_extract_images(dataset_path, processor)

### Text Preprocessing and Embedding Extraction

In [ ]:

def load_process_extract_images(dataset_path):
    text_embeddings = []
    text_ids = []

    for art in dataset_path:
        try:
            text_input = build_text_prompt(art)

            inputs = processor(
                text=text_input,
                return_tensors="pt",
                padding=True,
                truncation=True
            ).to(device)

            with torch.no_grad():
                features = model.get_text_features(**inputs)
                features = features / features.norm(dim=-1, keepdim=True)

            text_embeddings.append(features.cpu().numpy())
            text_ids.append(art.id)

        except Exception as e:
            print(f"Skipping text for {art.id}: {e}")

    return np.vstack(text_embeddings), text_ids

## Load CLIP Model and Extract Embeddings

The CLIP model contains 2 encoder Image and Text embeddings, respectively


In [ ]:


image_features = model.get_image_features(image_tensors)
model.get_text_features()

### Feature Fusion 

(1) Image Only (2) Text Only (3) Weighted Feature Fusion of Image and Text 

## Test Query 

Encode using text/image encoder or a hybrid of both 

In [ ]:
query = "A surreal painting with melting objects and dream-like atmosphere"

### Compute cosine similarity and Retreive top-k